### Link collab: https://drive.google.com/file/d/1-tl9hewReCwsHp0-NdGLpwxOLc-8Zj07/view?usp=sharing

### Link file pkl (có thể tải và dùng luôn, đặt vào thư mục notebooks/RA_Rec): https://drive.google.com/file/d/1KPPWXGv52jamRULpjw2EFVYMdx24IG2x/view?usp=drive_link

## 1. Setup & Load Libraries

In [1]:
import pandas as pd
import numpy as np
import ast
import pickle
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

## 2. Load Dataset

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# Load dataset
#all_recipes_df = pd.read_csv("../../data/all_recipes_final.csv")
all_recipes_df = pd.read_csv("/content/drive/MyDrive/Project/DS300/all_recipes_final.csv")

print(f"Loaded dataset: {len(all_recipes_df)} recipes")
print(f"Columns: {all_recipes_df.columns.tolist()}")

Loaded dataset: 10263 recipes
Columns: ['title', 'type_of_food', 'link', 'description', 'ingredients', 'ingredients_normalized', 'step', 'note', 'num_of_ingredients', 'cook_time', 'num_of_people', 'calories', 'source']


## 3. Hybrid Chunking Strategy

- Metadata = 1 sentence
- Nguyên liệu theo NHÓM (5 items/group) = 2-3 sentences (sử dụng ingredients_normalized)
- Bước nấu theo PHASE (3 steps/group) = 3-4 sentences
- Mô tả = 1 sentence

In [6]:
import ast

def parse_list_field(field_value):
    """
    Parse string / list field to Python list safely.
    """
    if pd.isna(field_value):
        return []

    if isinstance(field_value, list):
        return field_value

    if isinstance(field_value, str):
        try:
            parsed = ast.literal_eval(field_value)
            if isinstance(parsed, list):
                return parsed
            return []
        except (ValueError, SyntaxError):
            # fallback: split by comma
            return [s.strip() for s in field_value.split(",") if s.strip()]

    return []


def parse_set_field(field_value):
    """
    Parse string / set field to Python list safely.
    Specifically for ingredients_normalized which is stored as set.
    """
    if pd.isna(field_value):
        return []

    if isinstance(field_value, (list, set)):
        return list(field_value)

    if isinstance(field_value, str):
        try:
            parsed = ast.literal_eval(field_value)
            if isinstance(parsed, (list, set)):
                return list(parsed)
            return []
        except (ValueError, SyntaxError):
            # fallback: split by comma
            return [s.strip() for s in field_value.split(",") if s.strip()]

    return []

In [7]:
def hybrid_chunking(row):
    """
    Hybrid Chunking Strategy (Production-ready)

    Structure:
    - Metadata: 1 chunk
    - Ingredients: group ~5 items, avoid small tail chunks (ingredients_normalized)
    - Steps: group ~3 steps, avoid isolated steps
    - Description: 1 chunk
    """
    sentences = []

    # Metadata
    meta_parts = []

    if pd.notna(row.get("type_of_food")):
        meta_parts.append(str(row["type_of_food"]).strip())

    if pd.notna(row.get("title")):
        meta_parts.append(str(row["title"]).strip())

    if pd.notna(row.get("cook_time")):
        meta_parts.append(f"thời gian {row['cook_time']}")

    if pd.notna(row.get("num_of_people")):
        meta_parts.append(f"Số người ăn: {row['num_of_people']}")

    if meta_parts:
        sentences.append(". ".join(meta_parts))

    # 2. Ingredients (group by ~5, min 3)
    # Ưu tiên sử dụng ingredients_normalized, fallback sang ingredients nếu không có
    ingredients = parse_set_field(row.get("ingredients_normalized"))
    if not ingredients:
        ingredients = parse_list_field(row.get("ingredients"))

    if ingredients:
        chunk_size = 5
        min_chunk = 3

        i = 0
        while i < len(ingredients):
            # merge small tail into previous chunk
            if len(ingredients) - i < min_chunk and sentences:
                sentences[-1] += ", " + ", ".join(ingredients[i:])
                break

            chunk = ingredients[i:i + chunk_size]
            sentences.append("Nguyên liệu: " + ", ".join(chunk))
            i += chunk_size

    # 3. Steps (group by ~3, min 2)
    steps = parse_list_field(row.get("step"))

    if steps:
        chunk_size = 3
        min_chunk = 2

        i = 0
        while i < len(steps):
            # merge isolated tail steps
            if len(steps) - i < min_chunk and sentences:
                sentences[-1] += " → " + " → ".join(steps[i:])
                break

            chunk = steps[i:i + chunk_size]
            sentences.append(" → ".join(chunk))
            i += chunk_size

    # 4. Description
    if pd.notna(row.get("description")):
        desc = str(row["description"]).strip()
        if desc:
            sentences.append(desc)

    # 5. Notes / Tips (list[str] → single semantic chunk)
    notes = parse_list_field(row.get("note"))

    if notes:
        note_text = " | ".join(notes)
        sentences.append("Lưu ý: " + note_text)

    return sentences

In [9]:
test_df = all_recipes_df[24:25]

for index, row in test_df.iterrows():
    chunks = hybrid_chunking(row)
    print(f"Recipe: {index}")
    for i, chunk in enumerate(chunks):
        print(f" Chunk {i+1}: {chunk}")
    print("\n")

Recipe: 24
 Chunk 1: Món Tết. Gỏi ngũ sắc từ thịt gà thừa dịp Tết. thời gian 30 phút. Số người ăn: 4-6 người
 Chunk 2: Nguyên liệu: rau gia vị: rau răm, đường, tỏi, dưa chuột, quả đu đủ
 Chunk 3: Nguyên liệu: củ cả rốt, rau mùi, hạt tiêu, củ đậu, gia vị: mắm
 Chunk 4: Nguyên liệu: kinh giới tùy chọn, chanh, ớt, rau húng, muối
 Chunk 5: Nguyên liệu: hành khô phi, lạc rang giã dập, con gà luộc dư, bắp cải tím
 Chunk 6: Bước 1: Gà luộc còn dư đem xé hoặc thái miếng vừa ăn. Không xé nhỏ quá làm món ăn bị khô. Ướp gà với 1/2 muỗng cà phê bột canh (muối), 1/2 muỗng cà phê hạt tiêu, 1 muỗng cà phê đường, chút nước cốt chanh đảo đều cho thấm vị. → Bước 2: Tùy theo sở thích và nguyên liệu sẵn có mà biến tấu phần rau củ quả cho phù hợp. Đu đủ gọt vỏ, bào sợi. Cà rốt gọt bỏ vỏ, bào sợi. Củ đậu cắt sợi dài. Dưa chuột ngâm nước muối loãng lấy phần vỏ xanh cắt sợi dài hoặc bào sợi tùy chọn. Bắp cải tím cắt lát mỏng ngâm nước đá cho giòn. Lạc rang vàng thơm, xát bỏ vỏ, giã dập sơ. → Bước 3: Pha nước 

## 4. Generate Sentences for All Recipes

In [10]:
# Generate sentences for all recipes
all_recipes_sentences = []
for idx, row in tqdm(all_recipes_df.iterrows(), total=len(all_recipes_df), desc="Processing recipes"):
    sentences = hybrid_chunking(row)
    # hybrid_chunking on each row return list of sentences in one recipe
    all_recipes_sentences.append(sentences)

# Statistics
total_sentences = sum(len(s) for s in all_recipes_sentences)
sentences_per_recipes = [len(s) for s in all_recipes_sentences]

print(f"CHUNKING RESULTS:")
print(f"   Total recipes:        {len(all_recipes_sentences):,}")
print(f"   Total sentences:      {total_sentences:,}")
print(f"   Min sentences_per_recipes:           {min(sentences_per_recipes)}")
print(f"   Max sentences_per_recipes:           {max(sentences_per_recipes)}")
print(f"   Median sentences_per_recipes:        {np.median(sentences_per_recipes):.1f}")

Processing recipes: 100%|██████████| 10263/10263 [00:03<00:00, 3289.04it/s]

CHUNKING RESULTS:
   Total recipes:        10,263
   Total sentences:      69,156
   Min sentences_per_recipes:           3
   Max sentences_per_recipes:           23
   Median sentences_per_recipes:        7.0


In [11]:
all_recipes_sentences[:2]

[['Món Tết. Cách muối dưa hành truyền thống. thời gian 45 phút. Số người ăn: 8-10 người',
  'Nguyên liệu: hành củ tươi, lọ sạch, đường, muối hạt, tro bếp hoặc nước vo gọa, cà rốt trang trí tùy chọn',
  "Bước 1: Chọn hành củ: Nên chọn hành củ ta bánh tẻ, vừa phải, cầm chắc tay, tròn căng mọng, màu sắc tươi đều (tím nhạt hoặc trắng). Tránh mua hành ấn vào mềm, chảy nước hoặc mốc là đã hỏng. Chỉ nên lựa củ vừa phải, không nên to quá. → Bước 2: Ngâm khử mùi hăng của hành: Theo kinh nghiệm dân gian và trong sách ''Thế vị tân biên'' xuất bản năm 1925 đề cập để khử hăng, giúp hành giòn và trắng nên ngâm với nước tro bếp 2 - 3 ngày. Có nhà dùng nước vo gạo ngâm cũng có tác dụng tương tự. → Bước 3: Nhặt rễ, ngâm nước muối: Dùng dao nhỏ sắc, cắt gần sát rễ (không cắt hết), bóc lớp vỏ già ngoài. Sau đó, ngâm hành vào nước muối loãng ngâm khoảng 30 phút. Việc này giúp khử hành bớt hăng và khử khuẩn để khi ngâm hành không bị nổi váng, úng nhớt. Nếu muốn tăng thêm màu sắc bắt mắt, tỉa thêm chút hoa 

In [12]:
print(len(all_recipes_sentences))
print(type(all_recipes_sentences[1][1]))

10263
<class 'str'>


all_recipes_sentences = [[], [],...]

## 5. Load Vietnamese SBERT Model

In [13]:
# Load Vietnamese SBERT model
model = SentenceTransformer('keepitreal/vietnamese-sbert')
print(f"   Embedding dimension: {model.get_sentence_embedding_dimension()}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

bpe.codes: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/17.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

   Embedding dimension: 768


## 6. Encode All Sentences → Multi-vector Embeddings

In [14]:
def encode_multi_vector_dishes(all_recipes_sentences, model, batch_size=64):
    """
    Encode all sentences into embeddings

    Returns:
        recipes_embeddings_list: List of numpy arrays, mỗi món = list of vectors
    """
    print(f"Encoding {len(all_recipes_sentences)} recipes...")

    recipes_embeddings_list = []

    for recipe_sentences in tqdm(all_recipes_sentences, desc="Encoding recipes"):
        if len(recipe_sentences) == 0:
            recipes_embeddings_list.append(np.array([]))
            continue

        # Encode sentences for this recipe
        recipe_embeddings = model.encode(
            recipe_sentences,
            batch_size=batch_size,
            show_progress_bar=False
        )
        recipes_embeddings_list.append(recipe_embeddings)

    print(f"   {len(recipes_embeddings_list):,} recipes encoded")

    return recipes_embeddings_list

In [15]:
# Encode all dishes
recipes_embeddings_list = encode_multi_vector_dishes(
    all_recipes_sentences,
    model,
    batch_size=64
)

Encoding 10263 recipes...


Streaming output truncated to the last 5000 lines.
Encoding recipes: 100%|██████████| 10263/10263 [16:04<00:00, 10.64it/s]

   10,263 recipes encoded


In [16]:
print(f"EMBEDDING RESULTS:")
print(f"   Total recipes: {len(recipes_embeddings_list)}")
print(f"   Sample recipe 0: {recipes_embeddings_list[0].shape}")
print(f"   Sample recipe 1: {recipes_embeddings_list[1].shape}")

EMBEDDING RESULTS:
   Total recipes: 10263
   Sample recipe 0: (5, 768)
   Sample recipe 1: (7, 768)


## 7. Save Embeddings

In [17]:
import os
import pickle

path = "/content/drive/MyDrive/Project/DS300/recipes_embeddings_list.pkl"
os.makedirs(os.path.dirname(path), exist_ok=True)

with open(path, "wb") as f:
    pickle.dump(recipes_embeddings_list, f, protocol=pickle.HIGHEST_PROTOCOL)

print("Saved: recipes_embeddings_list.pkl")


Saved: recipes_embeddings_list.pkl
